In [1]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.constants import START, END
from langgraph.graph import MessagesState, StateGraph

load_dotenv()
# 获取模型
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


# 定义节点
def llm_node(state: MessagesState) -> MessagesState:
    messages = state["messages"]
    response = model.invoke(messages)

    return {
        "messages": [response]
    }


# 构建图
builder = StateGraph(state_schema=MessagesState)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

# 使用流式输出
for chunk in graph.invoke(
        {
            "messages": [HumanMessage(content="你好！")]
        },
        stream_mode=["values", "messages"]):
    print(chunk)

('values', {'messages': [HumanMessage(content='你好！', additional_kwargs={}, response_metadata={}, id='36a64a2d-2a8c-47fb-86c6-394771544845')]})
('messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--019fdb79-a57d-7423-b399-3a9d79a1a8bc', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'langgraph_step': 1, 'langgraph_node': 'llm_node', 'langgraph_triggers': ('branch:to:llm_node',), 'langgraph_path': ('__pregel_pull', 'llm_node'), 'langgraph_checkpoint_ns': 'llm_node:3be9ad90-e0ca-11c2-a692-f70a2ef14da2', 'checkpoint_ns': 'llm_node:3be9ad90-e0ca-11c2-a692-f70a2ef14da2', 'ls_provider': 'deepseek', 'ls_model_name': 'deepseek-v4-flash', 'ls_model_type': 'chat', 'ls_temperature': None}))
('messages', (AIMessageChunk(content='你好', additional_kwargs={}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--019fdb79-a57d-7423-b399-3a9d79a1a8bc', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {